# 10 – Single-Factor Benchmark Challenge

This notebook tests the two retained standalone factors against the three shortlisted portfolios under a controlled, common implementation framework. SPY remains a contextual long-only benchmark.

## Objective

The analysis asks whether the existing final hierarchy remains justified after adding **Momentum Only** and **Realised Volatility Only** as controlled component-factor benchmarks. Comparisons emphasise matched dates, frequencies, costs, rebalance phases, exposure budgets, and accounting conventions.

## Pre-registered expectations

Before inspecting the controlled results:

1. Composite Score is expected to remain preferable to Realised Volatility Only on combined return and risk evidence.
2. Realised Volatility Only may exceed the sleeve portfolios in raw return but is expected to carry greater risk or concentration.
3. Momentum Only is expected to remain weak as a standalone portfolio.
4. Momentum may still improve Composite Score through ranking interaction or diversification.
5. The existing hierarchy is expected to remain defensible, but contradictory evidence will be reported directly.

These are testable expectations, not target conclusions.

## Frozen research design

Factor definitions, directions, signal processing, quintiles, gross budgets, forward-return alignment, drift-aware holdings, full-L1 turnover, linear transaction costs, missing-return treatment, benchmark treatment, and numerical tolerance remain unchanged. The controlled window is **2016-01-07 to 2026-07-01**, and the baseline cost is **10 bps per unit of turnover**. No data are downloaded and no existing artifact is written.

## 1. Setup and input audits

The audit establishes file availability, schemas, unique keys, and the exact shared date index before any portfolio is reconstructed.

In [1]:
import numpy as np
import pandas as pd

from IPython.display import display

from alpha_research.backtest import (
    BacktestConfig,
    run_long_short_backtest,
    run_target_weight_backtest,
    summarise_backtest,
)
from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.config.research import (
    BACKTEST_RETURN_COLUMN,
    BASELINE_TRANSACTION_COST_BPS,
    DEFAULT_NUMERICAL_TOLERANCE,
    STRATEGY_EVALUATION_START_DATE,
    STRATEGY_SPECIFICATIONS,
)
from alpha_research.dashboard_analytics import prepare_performance_history
from alpha_research.data_loader import load_parquet
from alpha_research.metrics import summarise_returns
from alpha_research.monitoring import calculate_performance_risk_state
from alpha_research.visualisation import (
    build_cumulative_performance_figure,
    build_drawdown_figure,
)
from alpha_research.workflows import (
    build_common_strategy_backtests,
    build_frozen_strategy_target_weights,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

CHALLENGE_END_DATE = pd.Timestamp("2026-07-01")
AUDIT_TOLERANCE = DEFAULT_NUMERICAL_TOLERANCE
ACTIVE_PORTFOLIO_ORDER = (
    "Momentum Only",
    "Realised Volatility Only",
    "Composite Score",
    "Fixed 50/50 Sleeves",
    "Pure Inverse Volatility",
)
COMPARISON_ORDER = (*ACTIVE_PORTFOLIO_ORDER, "SPY")
LEGACY_NAME_MAP = {
    "12-1 Momentum": "Momentum Only",
    "Realised Volatility": "Realised Volatility Only",
}

In [2]:
ARTIFACT_PATHS = {
    "factor_panel": PROCESSED_DATA_DIR / "factor_panel.parquet",
    "factor_backtest_summary": PROCESSED_DATA_DIR / "factor_backtest_summary.parquet",
    "momentum_daily": PROCESSED_DATA_DIR / "backtest_12_1_momentum_daily.parquet",
    "momentum_holdings": PROCESSED_DATA_DIR / "backtest_12_1_momentum_holdings.parquet",
    "volatility_daily": PROCESSED_DATA_DIR / "backtest_realised_volatility_daily.parquet",
    "volatility_holdings": PROCESSED_DATA_DIR / "backtest_realised_volatility_holdings.parquet",
    "five_day_candidates": PROCESSED_DATA_DIR / "portfolio_optimisation_benchmarks.parquet",
    "selected_candidate_daily": PROCESSED_DATA_DIR / "attribution_portfolio_daily.parquet",
    "benchmark_daily": PROCESSED_DATA_DIR / "attribution_benchmark_daily.parquet",
}

missing_paths = [str(path) for path in ARTIFACT_PATHS.values() if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Required local artifacts are missing: {missing_paths}")

artifact_manifest = pd.DataFrame(
    [
        {"dataset": name, "size_bytes": path.stat().st_size}
        for name, path in ARTIFACT_PATHS.items()
    ]
)

factor_panel = load_parquet(ARTIFACT_PATHS["factor_panel"])
stored_factor_summary = load_parquet(ARTIFACT_PATHS["factor_backtest_summary"])
stored_legacy_daily = {
    "12-1 Momentum": load_parquet(ARTIFACT_PATHS["momentum_daily"]),
    "Realised Volatility": load_parquet(ARTIFACT_PATHS["volatility_daily"]),
}
stored_legacy_holdings = {
    "12-1 Momentum": load_parquet(ARTIFACT_PATHS["momentum_holdings"]),
    "Realised Volatility": load_parquet(ARTIFACT_PATHS["volatility_holdings"]),
}
five_day_candidate_artifact = load_parquet(ARTIFACT_PATHS["five_day_candidates"])
selected_candidate_daily = load_parquet(ARTIFACT_PATHS["selected_candidate_daily"])
benchmark_daily = load_parquet(ARTIFACT_PATHS["benchmark_daily"])

display(artifact_manifest)

,dataset,size_bytes
0,factor_panel,79031867
1,factor_backtest_summary,8469
2,momentum_daily,184309
3,momentum_holdings,92164
4,volatility_daily,195253
5,volatility_holdings,89457
6,five_day_candidates,520285
7,selected_candidate_daily,798207
8,benchmark_daily,47255


In [3]:
REQUIRED_FACTOR_COLUMNS = {
    "date",
    "ticker",
    BACKTEST_RETURN_COLUMN,
    "mom_12_1m_z",
    "realised_vol_63_z",
    "sector",
    "beta_126",
    "dollar_volume",
}
missing_factor_columns = REQUIRED_FACTOR_COLUMNS - set(factor_panel.columns)
if missing_factor_columns:
    raise KeyError(f"factor_panel is missing columns: {sorted(missing_factor_columns)}")

factor_panel["date"] = pd.to_datetime(factor_panel["date"], errors="raise")
five_day_candidate_artifact["date"] = pd.to_datetime(
    five_day_candidate_artifact["date"], errors="raise"
)
selected_candidate_daily["date"] = pd.to_datetime(
    selected_candidate_daily["date"], errors="raise"
)
benchmark_daily["date"] = pd.to_datetime(benchmark_daily["date"], errors="raise")

if factor_panel.duplicated(["date", "ticker"]).any():
    raise ValueError("factor_panel contains duplicate date-ticker keys.")
if five_day_candidate_artifact.duplicated(["portfolio", "date"]).any():
    raise ValueError("five_day_candidate_artifact contains duplicate keys.")
if selected_candidate_daily.duplicated(["portfolio", "date"]).any():
    raise ValueError("selected_candidate_daily contains duplicate keys.")
if benchmark_daily.duplicated(["benchmark", "date"]).any():
    raise ValueError("benchmark_daily contains duplicate keys.")
if benchmark_daily["benchmark"].drop_duplicates().tolist() != ["SPY"]:
    raise ValueError("Expected one SPY benchmark series.")

common_dates = pd.DatetimeIndex(benchmark_daily["date"]).sort_values()
if common_dates.min() != STRATEGY_EVALUATION_START_DATE:
    raise ValueError("Unexpected common-window start date.")
if common_dates.max() != CHALLENGE_END_DATE:
    raise ValueError("Unexpected common-window end date.")

required_candidate_names = set(ACTIVE_PORTFOLIO_ORDER[2:])
for source_name, source in {
    "five_day_candidates": five_day_candidate_artifact.loc[
        five_day_candidate_artifact["portfolio"].isin(required_candidate_names)
    ],
    "selected_candidates": selected_candidate_daily,
}.items():
    for portfolio, portfolio_data in source.groupby("portfolio", sort=False):
        dates = pd.DatetimeIndex(portfolio_data["date"]).sort_values()
        if not dates.equals(common_dates):
            raise ValueError(f"{source_name}: {portfolio} dates do not match the common window.")

input_audit_rows = [
    {
        "dataset": "factor_panel",
        "rows": len(factor_panel),
        "start_date": factor_panel["date"].min(),
        "end_date": factor_panel["date"].max(),
        "duplicate_keys": int(factor_panel.duplicated(["date", "ticker"]).sum()),
    },
    {
        "dataset": "benchmark_daily",
        "rows": len(benchmark_daily),
        "start_date": benchmark_daily["date"].min(),
        "end_date": benchmark_daily["date"].max(),
        "duplicate_keys": int(benchmark_daily.duplicated(["benchmark", "date"]).sum()),
    },
]
for factor_name, daily in stored_legacy_daily.items():
    daily["date"] = pd.to_datetime(daily["date"], errors="raise")
    input_audit_rows.append(
        {
            "dataset": f"legacy_daily: {factor_name}",
            "rows": len(daily),
            "start_date": daily["date"].min(),
            "end_date": daily["date"].max(),
            "duplicate_keys": int(daily["date"].duplicated().sum()),
        }
    )

input_audit = pd.DataFrame(input_audit_rows)
display(input_audit)
print(f"Common evaluation dates: {len(common_dates):,}")

,dataset,rows,start_date,end_date,duplicate_keys
0,factor_panel,284249,2015-01-02,2026-07-02,0
1,benchmark_daily,2635,2016-01-07,2026-07-01,0
2,legacy_daily: 12-1 Momentum,2890,2015-01-02,2026-07-01,0
3,legacy_daily: Realised Volatility,2890,2015-01-02,2026-07-01,0


Common evaluation dates: 2,635


### Setup findings

All required local inputs are present. Security and portfolio keys are unique, and the final SPY series defines 2,635 common trading dates from 2016-01-07 through 2026-07-01. The factor panel extends one row-date further because the last price date supplies the forward return realised after 2026-07-01.

## 2. Baseline reproduction

The first audit replays the original five-day standalone script on its full historical window. The second replays the three frozen selected implementations on the final common window.

In [4]:
legacy_factor_columns = {
    "12-1 Momentum": "mom_12_1m_z",
    "Realised Volatility": "realised_vol_63_z",
}
replayed_legacy_daily = {}
replayed_legacy_holdings = {}
replayed_summary_rows = []
for factor_name, factor_column in legacy_factor_columns.items():
    daily, holdings = run_long_short_backtest(
        factor_panel,
        factor_column=factor_column,
        config=BacktestConfig(),
    )
    replayed_legacy_daily[factor_name] = daily
    replayed_legacy_holdings[factor_name] = holdings
    summary = summarise_backtest(daily).iloc[0].to_dict()
    summary["factor"] = factor_name
    replayed_summary_rows.append(summary)
replayed_factor_summary = pd.DataFrame(replayed_summary_rows).set_index("factor")
stored_summary_indexed = stored_factor_summary.set_index("factor").sort_index()
replayed_summary_indexed = replayed_factor_summary.sort_index()
legacy_audit_rows = []

for factor_name in LEGACY_NAME_MAP:
    replayed_daily = replayed_legacy_daily[factor_name].sort_values("date").reset_index(drop=True)
    stored_daily = stored_legacy_daily[factor_name].sort_values("date").reset_index(drop=True)
    replayed_holdings = replayed_legacy_holdings[factor_name].sort_values(
        ["date", "ticker"]
    ).reset_index(drop=True)
    stored_holdings = stored_legacy_holdings[factor_name].sort_values(
        ["date", "ticker"]
    ).reset_index(drop=True)

    daily_numeric_columns = [
        column for column in stored_daily.columns if column not in {"date", "is_rebalance"}
    ]
    summary_columns = list(stored_summary_indexed.select_dtypes(include="number").columns)
    daily_difference = max(
        (replayed_daily[column] - stored_daily[column]).abs().max()
        for column in daily_numeric_columns
    )
    holdings_difference = (replayed_holdings["weight"] - stored_holdings["weight"]).abs().max()
    summary_difference = max(
        abs(
            replayed_summary_indexed.loc[factor_name, column]
            - stored_summary_indexed.loc[factor_name, column]
        )
        for column in summary_columns
    )
    keys_match = replayed_holdings[["date", "ticker"]].equals(
        stored_holdings[["date", "ticker"]]
    )
    dates_match = replayed_daily["date"].equals(stored_daily["date"])
    flags_match = replayed_daily["is_rebalance"].equals(stored_daily["is_rebalance"])

    legacy_audit_rows.append(
        {
            "portfolio": LEGACY_NAME_MAP[factor_name],
            "observations": len(replayed_daily),
            "start_date": replayed_daily["date"].min(),
            "end_date": replayed_daily["date"].max(),
            "maximum_daily_difference": daily_difference,
            "maximum_holdings_difference": holdings_difference,
            "maximum_summary_difference": summary_difference,
            "audit_passes": bool(
                keys_match
                and dates_match
                and flags_match
                and max(daily_difference, holdings_difference, summary_difference)
                < AUDIT_TOLERANCE
            ),
        }
    )

legacy_reproduction_audit = pd.DataFrame(legacy_audit_rows)
if not legacy_reproduction_audit["audit_passes"].all():
    raise ValueError("The original standalone backtests did not reproduce.")

original_standalone_headlines = (
    stored_factor_summary.assign(
        portfolio=lambda data: data["factor"].map(LEGACY_NAME_MAP)
    )
    .set_index("portfolio")
    .reindex(ACTIVE_PORTFOLIO_ORDER[:2])
    .reset_index()[
        [
            "portfolio",
            "observations",
            "annualised_return",
            "annualised_volatility",
            "sharpe_ratio",
            "max_drawdown",
        ]
    ]
)

display(legacy_reproduction_audit)
display(original_standalone_headlines.round(6))

,portfolio,observations,start_date,end_date,maximum_daily_difference,maximum_holdings_difference,maximum_summary_difference,audit_passes
0,Momentum Only,2890,2015-01-02,2026-07-01,0.0,0.0,0.0,True
1,Realised Volatility Only,2890,2015-01-02,2026-07-01,0.0,0.0,0.0,True


,portfolio,observations,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown
0,Momentum Only,2890.0,0.018785,0.211280,0.194719,-0.490626
1,Realised Volatility Only,2890.0,0.157869,0.239941,0.731003,-0.452815


In [5]:
frozen_targets = build_frozen_strategy_target_weights(factor_panel)
strategy_lookup = {item.portfolio: item for item in STRATEGY_SPECIFICATIONS}
candidate_reproduction_rows = []
replayed_frozen_daily = {}
CANDIDATE_AUDIT_COLUMNS = [
    "long_return",
    "short_return",
    "gross_return",
    "turnover",
    "transaction_cost",
    "net_return",
    "long_exposure",
    "short_exposure",
    "net_exposure",
    "gross_exposure",
    "missing_return_weight",
    "gross_cumulative_return",
    "net_cumulative_return",
]
return_panel = factor_panel[["date", "ticker", BACKTEST_RETURN_COLUMN]].copy()

for portfolio, targets in frozen_targets.items():
    specification = strategy_lookup[portfolio]
    replayed, _ = run_target_weight_backtest(
        return_panel,
        targets,
        transaction_cost_bps=specification.transaction_cost_bps,
    )
    replayed = replayed.loc[replayed["date"].isin(common_dates)].sort_values("date").reset_index(drop=True)
    replayed["gross_cumulative_return"] = (1.0 + replayed["gross_return"]).cumprod()
    replayed["net_cumulative_return"] = (1.0 + replayed["net_return"]).cumprod()
    reference = selected_candidate_daily.loc[
        selected_candidate_daily["portfolio"].eq(portfolio)
    ].sort_values("date").reset_index(drop=True)
    maximum_difference = max(
        (replayed[column] - reference[column]).abs().max()
        for column in CANDIDATE_AUDIT_COLUMNS
    )
    dates_match = replayed["date"].equals(reference["date"])
    flags_match = replayed["is_rebalance"].equals(reference["is_rebalance"])
    candidate_reproduction_rows.append(
        {
            "portfolio": portfolio,
            "rebalance_frequency": specification.rebalance_frequency,
            "rebalance_offset": specification.rebalance_offset,
            "observations": len(replayed),
            "maximum_absolute_difference": maximum_difference,
            "audit_passes": bool(
                dates_match and flags_match and maximum_difference < AUDIT_TOLERANCE
            ),
        }
    )
    replayed_frozen_daily[portfolio] = replayed

candidate_reproduction_audit = pd.DataFrame(candidate_reproduction_rows)
if not candidate_reproduction_audit["audit_passes"].all():
    raise ValueError("The frozen selected candidates did not reproduce.")

frozen_candidate_headlines = []
for portfolio in ACTIVE_PORTFOLIO_ORDER[2:]:
    summary = summarise_backtest(replayed_frozen_daily[portfolio]).iloc[0]
    frozen_candidate_headlines.append(
        {
            "portfolio": portfolio,
            "annualised_return": summary["annualised_return"],
            "annualised_volatility": summary["annualised_volatility"],
            "sharpe_ratio": summary["sharpe_ratio"],
            "max_drawdown": summary["max_drawdown"],
        }
    )

display(candidate_reproduction_audit)
display(pd.DataFrame(frozen_candidate_headlines).round(6))

,portfolio,rebalance_frequency,rebalance_offset,observations,maximum_absolute_difference,audit_passes
0,Composite Score,21,0,2635,0.0,True
1,Fixed 50/50 Sleeves,10,0,2635,0.0,True
2,Pure Inverse Volatility,10,0,2635,0.0,True


,portfolio,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown
0,Composite Score,0.161268,0.211671,0.812777,-0.292379
1,Fixed 50/50 Sleeves,0.108436,0.167656,0.698280,-0.256835
2,Pure Inverse Volatility,0.109593,0.162527,0.721505,-0.217693


### Reproduction findings

Both original standalone artifacts reproduce on their longer sample, and all three frozen candidates reproduce on the final common window within the project tolerance. The original standalone engine holds weights constant between rebalance dates; the controlled challenge below instead uses the final drift-aware target-weight engine for all five active portfolios. Restricting the legacy returns to the common dates would therefore not be an identical-framework test.

## 3. Common-window comparison

All five active portfolios are now reconstructed at a five-day frequency, offset zero, and 10 bps. SPY is included without an active-strategy cost model and should be interpreted as long-only market context.

In [6]:
COMMON_BASELINE_CONFIG = BacktestConfig(
    rebalance_frequency=5,
    rebalance_offset=0,
    transaction_cost_bps=BASELINE_TRANSACTION_COST_BPS,
)
common_daily_full, common_holdings_full, common_targets_full = (
    build_common_strategy_backtests(factor_panel, COMMON_BASELINE_CONFIG)
)

common_daily = {}
for portfolio in ACTIVE_PORTFOLIO_ORDER:
    daily = common_daily_full[portfolio].loc[
        common_daily_full[portfolio]["date"].isin(common_dates)
    ].sort_values("date").reset_index(drop=True)
    daily["gross_cumulative_return"] = (1.0 + daily["gross_return"]).cumprod()
    daily["net_cumulative_return"] = (1.0 + daily["net_return"]).cumprod()
    common_daily[portfolio] = daily

common_baseline_daily = pd.concat(
    [
        common_daily[portfolio].assign(
            portfolio=portfolio,
            rebalance_frequency=COMMON_BASELINE_CONFIG.rebalance_frequency,
            rebalance_offset=COMMON_BASELINE_CONFIG.rebalance_offset,
            transaction_cost_bps=COMMON_BASELINE_CONFIG.transaction_cost_bps,
        )
        for portfolio in ACTIVE_PORTFOLIO_ORDER
    ],
    ignore_index=True,
).sort_values(["portfolio", "date"]).reset_index(drop=True)

print(f"Controlled baseline rows: {len(common_baseline_daily):,}")

Controlled baseline rows: 13,175


In [7]:
target_audit_rows = []
accounting_audit_rows = []
five_day_reproduction_rows = []

for portfolio in ACTIVE_PORTFOLIO_ORDER:
    daily = common_daily[portfolio]
    holdings = common_holdings_full[portfolio].loc[
        common_holdings_full[portfolio]["date"].isin(common_dates)
    ].copy()
    targets = common_targets_full[portfolio].loc[
        common_targets_full[portfolio]["date"].isin(common_dates)
    ].copy()

    target_state = targets.groupby("date")["weight"].agg(
        long_gross=lambda weights: weights.clip(lower=0.0).sum(),
        short_gross=lambda weights: -weights.clip(upper=0.0).sum(),
        net_exposure="sum",
        gross_exposure=lambda weights: weights.abs().sum(),
    )
    exact_budget_required = portfolio in ACTIVE_PORTFOLIO_ORDER[:3]
    exact_budget_passes = bool(
        not exact_budget_required
        or (
            np.allclose(target_state["long_gross"], 1.0, atol=AUDIT_TOLERANCE)
            and np.allclose(target_state["short_gross"], 1.0, atol=AUDIT_TOLERANCE)
        )
    )
    target_audit_rows.append(
        {
            "portfolio": portfolio,
            "rebalance_dates": targets["date"].nunique(),
            "maximum_absolute_net_target": target_state["net_exposure"].abs().max(),
            "minimum_target_gross": target_state["gross_exposure"].min(),
            "maximum_target_gross": target_state["gross_exposure"].max(),
            "audit_passes": bool(
                not targets.duplicated(["date", "ticker"]).any()
                and target_state["net_exposure"].abs().max() < AUDIT_TOLERANCE
                and target_state["gross_exposure"].le(2.0 + AUDIT_TOLERANCE).all()
                and exact_budget_passes
            ),
        }
    )

    holdings_state = (
        holdings.assign(
            long_weight=lambda data: data["weight"].clip(lower=0.0),
            short_weight=lambda data: -data["weight"].clip(upper=0.0),
            absolute_trade=lambda data: data["trade"].abs(),
        )
        .groupby("date")
        .agg(
            holdings_long_exposure=("long_weight", "sum"),
            holdings_short_exposure=("short_weight", "sum"),
            holdings_turnover=("absolute_trade", "sum"),
        )
        .reset_index()
    )
    reconciliation = daily.merge(holdings_state, on="date", validate="one_to_one")
    maximum_accounting_difference = max(
        (daily["long_return"] + daily["short_return"] - daily["gross_return"]).abs().max(),
        (daily["gross_return"] - daily["transaction_cost"] - daily["net_return"]).abs().max(),
        (
            daily["turnover"] * COMMON_BASELINE_CONFIG.transaction_cost_bps / 10_000.0
            - daily["transaction_cost"]
        ).abs().max(),
        (reconciliation["long_exposure"] - reconciliation["holdings_long_exposure"]).abs().max(),
        (reconciliation["short_exposure"] - reconciliation["holdings_short_exposure"]).abs().max(),
        (reconciliation["turnover"] - reconciliation["holdings_turnover"]).abs().max(),
    )
    accounting_audit_rows.append(
        {
            "portfolio": portfolio,
            "observations": len(daily),
            "maximum_absolute_difference": maximum_accounting_difference,
            "maximum_missing_return_weight": daily["missing_return_weight"].max(),
            "audit_passes": bool(
                pd.DatetimeIndex(daily["date"]).equals(common_dates)
                and not holdings.duplicated(["date", "ticker"]).any()
                and maximum_accounting_difference < AUDIT_TOLERANCE
                and daily["missing_return_weight"].max() < AUDIT_TOLERANCE
            ),
        }
    )

    if portfolio in required_candidate_names:
        reference = five_day_candidate_artifact.loc[
            five_day_candidate_artifact["portfolio"].eq(portfolio)
        ].sort_values("date").reset_index(drop=True)
        audit_columns = [
            "gross_return",
            "net_return",
            "turnover",
            "transaction_cost",
            "missing_return_weight",
            "gross_exposure",
            "net_exposure",
        ]
        maximum_difference = max(
            (daily[column] - reference[column]).abs().max() for column in audit_columns
        )
        five_day_reproduction_rows.append(
            {
                "portfolio": portfolio,
                "maximum_absolute_difference": maximum_difference,
                "rebalance_flags_match": daily["is_rebalance"].equals(reference["is_rebalance"]),
                "audit_passes": bool(
                    maximum_difference < AUDIT_TOLERANCE
                    and daily["is_rebalance"].equals(reference["is_rebalance"])
                ),
            }
        )

target_weight_audit = pd.DataFrame(target_audit_rows)
accounting_audit = pd.DataFrame(accounting_audit_rows)
five_day_candidate_reproduction_audit = pd.DataFrame(five_day_reproduction_rows)

if not target_weight_audit["audit_passes"].all():
    raise ValueError("Target-weight constraints failed.")
if not accounting_audit["audit_passes"].all():
    raise ValueError("Portfolio accounting failed.")
if not five_day_candidate_reproduction_audit["audit_passes"].all():
    raise ValueError("The controlled candidates do not reproduce the five-day artifact.")

display(target_weight_audit)
display(accounting_audit)
display(five_day_candidate_reproduction_audit)

,portfolio,rebalance_dates,maximum_absolute_net_target,minimum_target_gross,maximum_target_gross,audit_passes
0,Momentum Only,527,0.000000e+00,2.000000,2.0,True
1,Realised Volatility Only,527,0.000000e+00,2.000000,2.0,True
2,Composite Score,527,0.000000e+00,2.000000,2.0,True
3,Fixed 50/50 Sleeves,527,0.000000e+00,0.450000,2.0,True
4,Pure Inverse Volatility,527,4.857226e-17,0.485807,2.0,True


,portfolio,observations,maximum_absolute_difference,maximum_missing_return_weight,audit_passes
0,Momentum Only,2635,2.220446e-16,0.0,True
1,Realised Volatility Only,2635,2.220446e-16,0.0,True
2,Composite Score,2635,2.220446e-16,0.0,True
3,Fixed 50/50 Sleeves,2635,2.220446e-16,0.0,True
4,Pure Inverse Volatility,2635,2.220446e-16,0.0,True


,portfolio,maximum_absolute_difference,rebalance_flags_match,audit_passes
0,Composite Score,0.000000e+00,True,True
1,Fixed 50/50 Sleeves,0.000000e+00,True,True
2,Pure Inverse Volatility,1.110223e-15,True,True


In [8]:
baseline_rows = []
for portfolio in ACTIVE_PORTFOLIO_ORDER:
    daily = common_daily[portfolio]
    net_summary = summarise_backtest(daily, return_column="net_return").iloc[0]
    gross_summary = summarise_backtest(daily, return_column="gross_return").iloc[0]
    baseline_rows.append(
        {
            "portfolio": portfolio,
            "observations": int(net_summary["observations"]),
            "total_return": net_summary["total_return"],
            "annualised_return": net_summary["annualised_return"],
            "annualised_volatility": net_summary["annualised_volatility"],
            "sharpe_ratio": net_summary["sharpe_ratio"],
            "max_drawdown": net_summary["max_drawdown"],
            "positive_day_fraction": net_summary["positive_day_fraction"],
            "average_daily_turnover": net_summary["average_daily_turnover"],
            "average_rebalance_turnover": net_summary["average_rebalance_turnover"],
            "cumulative_transaction_cost": net_summary["total_transaction_cost"],
            "annualised_return_cost_drag": (
                gross_summary["annualised_return"] - net_summary["annualised_return"]
            ),
            "average_gross_exposure": daily["gross_exposure"].mean(),
            "average_net_exposure": daily["net_exposure"].mean(),
        }
    )

spy_summary = summarise_returns(benchmark_daily["benchmark_return"])
baseline_rows.append(
    {
        "portfolio": "SPY",
        "observations": int(spy_summary["observations"]),
        "total_return": spy_summary["total_return"],
        "annualised_return": spy_summary["annualised_return"],
        "annualised_volatility": spy_summary["annualised_volatility"],
        "sharpe_ratio": spy_summary["sharpe_ratio"],
        "max_drawdown": spy_summary["maximum_drawdown"],
        "positive_day_fraction": spy_summary["positive_day_fraction"],
        "average_daily_turnover": 0.0,
        "average_rebalance_turnover": np.nan,
        "cumulative_transaction_cost": 0.0,
        "annualised_return_cost_drag": 0.0,
        "average_gross_exposure": 1.0,
        "average_net_exposure": 1.0,
    }
)

common_window_baseline = (
    pd.DataFrame(baseline_rows)
    .set_index("portfolio")
    .reindex(COMPARISON_ORDER)
    .reset_index()
)

display(common_window_baseline.round(6))

,portfolio,observations,total_return,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown,positive_day_fraction,average_daily_turnover,average_rebalance_turnover,cumulative_transaction_cost,annualised_return_cost_drag,average_gross_exposure,average_net_exposure
0,Momentum Only,2635,0.103780,0.009488,0.221888,0.154481,-0.509550,0.537381,0.111424,0.557119,0.293602,0.028744,2.002969,0.000579
1,Realised Volatility Only,2635,3.756356,0.160838,0.248639,0.724266,-0.458528,0.530930,0.082102,0.410509,0.216338,0.024269,2.000736,0.001164
2,Composite Score,2635,2.841731,0.137370,0.213330,0.710582,-0.306277,0.552182,0.104652,0.523261,0.275758,0.030386,2.000719,0.001304
3,Fixed 50/50 Sleeves,2635,1.801410,0.103533,0.168423,0.669545,-0.256562,0.544972,0.087102,0.435508,0.229513,0.024487,1.541543,0.001087
4,Pure Inverse Volatility,2635,1.816898,0.104115,0.162893,0.689863,-0.195652,0.551044,0.092385,0.461923,0.243433,0.026005,1.607862,0.001077
5,SPY,2635,3.533806,0.155530,0.178295,0.900386,-0.337173,0.554459,0.000000,NaN,0.000000,0.000000,1.000000,1.000000


In [9]:
performance_risk_state = calculate_performance_risk_state(
    common_baseline_daily[["date", "portfolio", "net_return"]],
    benchmark_daily[["date", "benchmark_return"]],
    portfolios=ACTIVE_PORTFOLIO_ORDER,
)
performance_history = prepare_performance_history(
    performance_risk_state,
    portfolios=COMPARISON_ORDER,
)

display(
    build_cumulative_performance_figure(
        performance_history,
        title="Common-Window Cumulative Performance – Five-Day Rebalancing",
    )
)
display(
    build_drawdown_figure(
        performance_history,
        title="Common-Window Drawdown – Five-Day Rebalancing",
    )
)

### Common-window findings

- **Realised Volatility leads the controlled strategy set:** 16.08% annualised return and 0.724 Sharpe, versus 0.95% and 0.154 for Momentum.
- **The blends trade return for risk control:** Composite reaches 13.74% annualised return with a 0.711 Sharpe; Fixed 50/50 and Pure Inverse Volatility return about 10.4%, with lower volatility and shallower drawdowns.
- **Pure Inverse Volatility has the shallowest strategy drawdown:** -19.57%, compared with -25.66% for Fixed 50/50, -30.63% for Composite, and -45.85% for Realised Volatility.
- **Trading costs remain material:** the 10 bps assumption reduces annualised returns by 2.43-3.04 percentage points across the five strategies.
- **SPY is a reference, not a like-for-like strategy:** it records 15.55% annualised return, a 0.900 Sharpe, and a -33.72% maximum drawdown without simulated transaction costs.

These results establish the controlled baseline. Frequency, cost, and market-phase robustness are evaluated in the next round before drawing a final conclusion.